# Análisis NLP de los Planes Nacionales de Desarrollo (PND) de Colombia, 2006-2026

### **A. Rubiano Carlos Andrés**
### **Barrios Martínez Alexandra**

Este cuaderno construye un corpus con los PND de cinco gobiernos, lo segmenta en párrafos, lo etiqueta por temas con un diccionario de palabras clave y entrena clasificadores de temas. Las etiquetas iniciales son automáticas (supervisión débil); su evaluación no equivale a una validación humana.

**Pregunta de investigación:** ¿Cómo cambia la presencia relativa de educación, vivienda y empleo en los PND entre 2006 y 2026? Estos tres temas constituyen un foco exploratorio; se conserva el modelo de 18 temas como referencia. La comparación mide presencia textual, no presupuesto, ejecución ni impacto.

**Versión corregida:** ejecutar todas las celdas desde un entorno limpio. Las salidas anteriores se eliminaron porque deben recalcularse.

Preparación: instalar `requirements.txt` y ejecutar desde la raíz del repositorio. Se requieren conexión al DNP y los recursos de NLTK/spaCy indicados más adelante.

**Contenido**

| Parte | Sección | Resultado principal |
|---|---|---|
| I | Obtención de datos: scraping y descarga | PDF de los PND en `pnd_seleccion/` |
| II | Extracción de texto de los PDF | `.txt` por documento, `corpus_paginas.jsonl` y reporte de calidad |
| III | Segmentación en párrafos | `parrafos.jsonl` sin tablas, índices ni encabezados |
| IV | Etiquetado temático por diccionario | `segmentos_clasificados.csv/.xlsx` |
| V | Preparación de texto (preprocesamiento) | `segm_clasificados_limpios.csv` |
| VI | Distribución de clases y exploración semántica | Gráficas, POS, NER y nubes de palabras |
| VII | Modelado: vectorización TF-IDF y clasificación | Regresión logística, métricas y análisis de errores |

# Parte I · Obtención de datos: scraping y descarga

Se visitan las páginas del DNP de los cinco gobiernos objetivo (2006-2010, 2010-2014, 2014-2018, 2018-2022 y 2022-2026), se identifican los PDF de cada Plan Nacional de Desarrollo y se descargan los tomos seleccionados.

In [ ]:
# Utilidades estándar: sistema de archivos, expresiones regulares y pausas
import os, re, time
# Peticiones HTTP y lectura de HTML (to parse)
import requests
import urllib3
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote
from requests.exceptions import SSLError

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

from dnp_utils import limitar_segmentos, preparar_texto, firma_texto, muestra_validacion, reservar_revision, nuevo_modelo, separar_periodo, leer_consenso, acuerdo_revisores
import hashlib, json
from pathlib import Path

## I.1 Configuración, scraping, filtrado y descarga

Esta celda hace todo el proceso en cuatro pasos:

1. **Gobiernos:** lee la página de históricos y detecta los enlaces de cada periodo.
2. **PDF de cada gobierno:** entra a la página de cada periodo y recolecta los enlaces `.pdf`.
3. **Filtrado:** se queda solo con los tomos definidos en `SELECCION`.
4. **Descarga:** guarda cada PDF en una carpeta por gobierno y valida que el archivo sea un PDF real.

In [ ]:

# ================== CONFIGURACIÓN ==================
# Carpeta donde se guardarán los PDF descargados
CARPETA_BASE = "pnd_seleccion"

# Periodo de gobierno -> nombres de archivo (sin .pdf) de los tomos que se quieren descargar
SELECCION = {
    "2022-2026": ["plan-nacional-de-desarrollo-2022-2026-colombia-potencia-mundial-de-la-vida"],
    "2018-2022": ["tomo-1-dnp", "tomo-2-pnd"],
    "2014-2018": ["PND 2014-2018 Tomo 1 internet", "PND 2014-2018 Tomo 2 internet"],
    "2010-2014": ["PND2010-2014 Tomo I CD", "PND2010-2014 Tomo II CD"],
    "2006-2010": ["PND_Tomo_1", "PND_Tomo_2"],
}

# Listado histórico de PND y página del PND vigente (que no aparece en "Históricos")
URL_BASE = "https://www.dnp.gov.co/plan-nacional-desarrollo/Paginas/default.aspx"
URL_PND_ACTUAL = "https://www.dnp.gov.co/plan-nacional-desarrollo/Paginas/pnd-2022-2026.aspx"
# User-Agent de navegador para evitar bloqueos del servidor
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# ================== FUNCIONES ==================
# Sesión reutilizable: conserva cabeceras y conexiones entre peticiones
session = requests.Session()
session.headers.update(HEADERS)

# Elimina caracteres invisibles (zero-width, BOM) y espacios sobrantes
def limpiar(texto):
    return re.sub(r"[\u200b\u200c\u200d\ufeff]", "", texto).strip()

# Obtiene el nombre del archivo desde la URL, decodificado y sin la extensión .pdf
def nombre_sin_ext(url):
    return unquote(url.split("/")[-1]).rsplit(".pdf", 1)[0].strip()

def get(url, **kw):
    """GET con verificación de certificados y timeout; no degrada a SSL inseguro."""
    r = session.get(url, timeout=60, **kw)
    r.raise_for_status()
    return r

def get_soup(url):
    return BeautifulSoup(get(url).text, "html.parser")

# Descarga el PDF por bloques (streaming) y valida que realmente sea un PDF
def descargar(url, ruta):
    r = get(url, stream=True)
    with open(ruta, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 64):
            f.write(chunk)
    # Un PDF válido empieza con la firma %PDF; si no, se borra el archivo y se lanza un error
    with open(ruta, "rb") as f:
        if not f.read(5).startswith(b"%PDF"):
            os.remove(ruta)
            raise ValueError("El archivo descargado no es un PDF válido")

# ================== 1) GOBIERNOS ==================
# Carga la página principal y busca enlaces con el formato "AAAA-AAAA - Presidente"
soup = get_soup(URL_BASE)
# Patrón del periodo de gobierno, p. ej. "2018-2022 - "
patron = re.compile(r"^(\d{4})\s*-\s*(\d{4})\s*-\s*")

# El PND actual se agrega a mano porque no está en la sección de históricos
gobiernos = [("2022-2026 - Gustavo Petro", URL_PND_ACTUAL)]   # no está en "Históricos"
# Recorre los enlaces de la página y conserva los que corresponden a un gobierno
for a in soup.find_all("a", href=True):
    texto = re.sub(r"\s+", " ", limpiar(a.get_text(" ", strip=True)))
    if patron.match(texto):
        gobiernos.append((texto, urljoin(URL_BASE, limpiar(a["href"]))))
# Elimina duplicados conservando el orden
gobiernos = list(dict.fromkeys(gobiernos))

# Solo los periodos que necesitas
gobiernos = [(g, u) for g, u in gobiernos if g[:9] in SELECCION]
print("Gobiernos a revisar:", [g for g, _ in gobiernos])

# ================== 2) PDFs DE CADA GOBIERNO ==================
# Para cada gobierno, recolecta los enlaces a PDF de su página
resultados = []
for gobierno, url_gob in gobiernos:
    try:
        s = get_soup(url_gob)
    except Exception as e:
        print(f"[ERROR] {gobierno}: {e}")
        continue
    # "vistos" evita registrar el mismo PDF dos veces
    vistos = set()
    for a in s.find_all("a", href=True):
        href = limpiar(a["href"])
        if ".pdf" not in href.lower():
            continue
        url_pdf = urljoin(url_gob, href)
        if url_pdf in vistos:
            continue
        vistos.add(url_pdf)
        resultados.append({"gobierno": gobierno, "url_pdf": url_pdf})
    print(f"{gobierno}: {len(vistos)} PDF(s)")
    # Pausa entre peticiones para no saturar el servidor
    time.sleep(1)

# ================== 3) FILTRAR ==================
# Conserva solo los PDF cuyo nombre coincide con los definidos en SELECCION
elegidos, encontrados = [], set()
for r in resultados:
    periodo = r["gobierno"][:9]
    nombre = nombre_sin_ext(r["url_pdf"])
    for buscado in SELECCION.get(periodo, []):
        if nombre.lower() == buscado.lower() and (periodo, buscado) not in encontrados:
            elegidos.append(r)
            encontrados.add((periodo, buscado))

# Avisa qué documentos de la selección no se encontraron en las páginas
for periodo, nombres in SELECCION.items():
    for n in nombres:
        if (periodo, n) not in encontrados:
            print(f"[NO ENCONTRADO] {periodo} -> {n}")

print(f"\nSeleccionados: {len(elegidos)} de {sum(len(v) for v in SELECCION.values())}")

# ================== 4) DESCARGAR ==================
# Crea una carpeta por gobierno y descarga cada PDF (omite los que ya existen)
fallidos = []
manifiesto = []
for i, r in enumerate(elegidos, 1):
    carpeta = os.path.join(CARPETA_BASE, re.sub(r'[\\/:*?"<>|]', "", r["gobierno"]))
    os.makedirs(carpeta, exist_ok=True)
    nombre = re.sub(r'[\\/:*?"<>|]', "_", unquote(r["url_pdf"].split("/")[-1]))
    ruta = os.path.join(carpeta, nombre)

    manifiesto.append({"periodo": r["gobierno"][:9], "url_pdf": r["url_pdf"], "ruta": ruta})
    if os.path.exists(ruta):
        print(f"[{i}/{len(elegidos)}] ya existe: {ruta}")
        continue
    try:
        descargar(r["url_pdf"], ruta)
        print(f"[{i}/{len(elegidos)}] OK  {ruta}")
    except Exception as e:
        print(f"[{i}/{len(elegidos)}] ERROR {r['url_pdf']}\n    {e}")
        fallidos.append(r)
    # Pausa entre descargas
    time.sleep(1)

print(f"\nDescargados: {len(elegidos) - len(fallidos)} | Fallidos: {len(fallidos)}")
print("Ruta:", os.path.abspath(CARPETA_BASE))
# Huellas permiten detectar cambios de fuente entre ejecuciones.
for registro in manifiesto:
    archivo = Path(registro["ruta"])
    registro["sha256"] = hashlib.sha256(archivo.read_bytes()).hexdigest() if archivo.exists() else None
Path(CARPETA_BASE, "manifiesto_fuentes.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")
if fallidos or len(elegidos) != sum(map(len, SELECCION.values())):
    raise RuntimeError("Corpus incompleto: revisar documentos faltantes antes de continuar")


# Parte II · Extracción de texto de los PDF

Se convierte cada PDF en texto limpio y se genera un reporte de calidad para detectar documentos problemáticos (por ejemplo, escaneados).

In [ ]:
%pip install -r requirements.txt
# Entorno de referencia: Python 3.11; ver README para recursos de spaCy/NLTK.


## II.1 Extracción, limpieza y reporte de calidad

Para cada PDF se extrae el texto página por página con PyMuPDF, se limpia (guiones de fin de línea, saltos de línea dentro de párrafos, encabezados y pies repetidos) y se guardan:

- un `.txt` por documento, con marcadores `[PÁGINA n]`,
- `corpus_paginas.jsonl`, con un registro por página y sus metadatos,
- `reporte_calidad.csv`, con métricas por documento (páginas sin texto, caracteres, líneas repetidas eliminadas).

In [ ]:
# PyMuPDF lee los PDF; pandas arma el reporte de calidad
import re, json
from pathlib import Path
from collections import Counter
import pymupdf          # PyMuPDF (en versiones antiguas: import fitz)
import pandas as pd

# ---------- Configuración ----------
CARPETA_PDFS = Path("pnd_seleccion")   # la misma carpeta de la descarga
CARPETA_CORPUS = Path("corpus")
CARPETA_CORPUS.mkdir(exist_ok=True)
(CARPETA_CORPUS / "txt").mkdir(exist_ok=True)

MIN_CARACTERES = 50   # página con menos texto que esto se marca como "vacía"

# ---------- Limpieza ----------
# Limpia el texto de una página: quita guiones suaves, une palabras partidas y compacta espacios
def limpiar_texto(t):
    t = t.replace("\u00ad", "")                  # guion suave
    t = re.sub(r"-\n(?=[a-záéíóúñü])", "", t)    # palabras cortadas por guion al final de línea
    t = re.sub(r"(?<![.:;!?\n])\n(?!\n)", " ", t)  # une líneas dentro de un mismo párrafo
    # Colapsa espacios y limita los saltos de línea a un solo párrafo en blanco
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

# Encuentra líneas que se repiten en muchas páginas (encabezados y pies de página)
def lineas_repetidas(paginas_texto, umbral=0.4):
    """Detecta encabezados/pies de página: líneas cortas que se repiten en muchas páginas."""
    n = len(paginas_texto)
    if n < 5:
        return set()
    cont = Counter()
    for t in paginas_texto:
        for linea in {l.strip() for l in t.split("\n") if 0 < len(l.strip()) < 80}:
            cont[linea] += 1
    return {l for l, c in cont.items() if c / n >= umbral}

# ---------- Metadatos desde el nombre de la carpeta ----------
# Separa periodo y presidente a partir del nombre de la carpeta
def parse_gobierno(nombre_carpeta):
    # "2018-2022 - Iván Duque Márquez"
    m = re.match(r"(\d{4}-\d{4})\s*-\s*(.+)", nombre_carpeta)
    return (m.group(1), m.group(2).strip()) if m else (nombre_carpeta, "")

# ---------- Procesamiento ----------
registros = []   # un dict por página
reporte = []     # un dict por documento

# Recorre todos los PDF: extrae, limpia y registra cada página
for pdf_path in sorted(CARPETA_PDFS.rglob("*.pdf")):
    periodo, presidente = parse_gobierno(pdf_path.parent.name)

    # sort=True ordena los bloques de texto en orden de lectura
    with pymupdf.open(pdf_path) as doc:
        crudo = [p.get_text("text", sort=True) for p in doc]

    # Detecta los encabezados/pies del documento para eliminarlos
    repetidas = lineas_repetidas(crudo)
    paginas_vacias = 0
    textos_doc = []

    # Limpia cada página y guarda un registro con sus metadatos
    for i, t in enumerate(crudo, 1):
        t = "\n".join(l for l in t.split("\n") if l.strip() not in repetidas)
        t = limpiar_texto(t)
        # Cuenta las páginas casi sin texto (posibles escaneos, portadas o páginas en blanco)
        if len(t) < MIN_CARACTERES:
            paginas_vacias += 1
        registros.append({
            "periodo": periodo,
            "presidente": presidente,
            "documento": pdf_path.stem,
            "pagina": i,
            "n_caracteres": len(t),
            "texto": t,
        })
        textos_doc.append(t)

    # .txt por documento, con marcador de página
    salida = CARPETA_CORPUS / "txt" / f"{periodo}__{pdf_path.stem}.txt"
    salida.write_text(
        "\n\n".join(f"[PÁGINA {i}]\n{t}" for i, t in enumerate(textos_doc, 1)),
        encoding="utf-8",
    )

    # Métricas de calidad por documento
    reporte.append({
        "periodo": periodo,
        "documento": pdf_path.stem,
        "paginas": len(crudo),
        "paginas_sin_texto": paginas_vacias,
        "pct_sin_texto": round(100 * paginas_vacias / max(len(crudo), 1), 1),
        "caracteres": sum(len(t) for t in textos_doc),
        "lineas_repetidas_eliminadas": len(repetidas),
    })
    print(f"✔ {periodo} | {pdf_path.name} | {len(crudo)} págs | {paginas_vacias} sin texto")

# ---------- Guardar ----------
# JSONL: un registro JSON por línea (una página por registro)
with open(CARPETA_CORPUS / "corpus_paginas.jsonl", "w", encoding="utf-8") as f:
    for r in registros:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df_reporte = pd.DataFrame(reporte)
df_reporte.to_csv(CARPETA_CORPUS / "reporte_calidad.csv", index=False, encoding="utf-8-sig")
print("\n", df_reporte.to_string(index=False))
print("\nCorpus en:", CARPETA_CORPUS.resolve())

## II.2 Verificación rápida del corpus

Se carga el corpus por páginas y se cuentan las palabras por gobierno para comprobar que la extracción funcionó.

In [ ]:
# Carga el corpus por páginas (un registro por línea)
df = pd.read_json("corpus/corpus_paginas.jsonl", lines=True)

# Texto completo de un gobierno
texto_duque = "\n".join(df[df.periodo == "2018-2022"].texto)

# Palabras por gobierno
# Conteo de palabras por página y total por gobierno
df["n_palabras"] = df.texto.str.split().str.len()
df.groupby("periodo").n_palabras.sum()

# Parte III · Segmentación del corpus en párrafos

A partir de los `.txt` se construyen segmentos de texto aptos para el análisis. Las reglas de depuración se describen en la siguiente celda.

## III.1 Detección de tablas, índices y encabezados, y segmentación

Esta celda aplica las reglas anteriores en tres etapas:

1. **Exclusión de páginas:** descarta páginas de índice (puntos guía, encabezados de índice) y páginas de tablas (heurística de proporción numérica y escasez de oraciones).
2. **Limpieza de bloques:** elimina encabezados y pies repetidos, números de página sueltos y bloques de menos de 20 palabras.
3. **Segmentación:** conserva los párrafos y divide por oraciones los que superan 2.000 caracteres.

Salidas: `parrafos.jsonl`, `paginas_excluidas.csv` y `resumen_segmentacion.csv`.
**Corrección:** por defecto las páginas candidatas a tabla se conservan y se exportan para auditoría. Los segmentos unidos registran página inicial y final; el límite de 2.000 caracteres se verifica explícitamente. Los fragmentos finales pueden tener menos de 20 palabras para evitar eliminar contenido.


In [ ]:
# math.ceil se usa para repartir párrafos largos en trozos parejos
import re, json, math
from pathlib import Path
from collections import Counter
import pandas as pd

# ================== CONFIGURACIÓN ==================
CARPETA_CORPUS = Path("corpus")
CARPETA_TXT    = CARPETA_CORPUS / "txt"
CARPETA_PDFS   = Path("pnd_seleccion")

# Parámetros de segmentación
MIN_PALABRAS   = 20      # bloques con menos palabras se descartan
MAX_CARACTERES = 2000    # párrafos más largos se dividen por oraciones

# Detección de tablas (heurística sobre el texto)
UMBRAL_NUMERICO  = 0.35  # % de tokens numéricos para marcar tabla directamente
UMBRAL_NUM_MIXTO = 0.15  # % numérico mínimo si además casi no hay oraciones
MAX_PUNTOS_X100  = 1.5   # puntos finales por cada 100 palabras (prosa suele tener 4-8)
EXCLUIR_PAGINAS_TABLA = False  # conservar prosa hasta auditar las candidatas
USAR_PDF_TABLAS  = False # True: refuerza con PyMuPDF find_tables() (más lento)

# Encabezados / pies repetidos
UMBRAL_REPETIDO  = 0.10  # un bloque en el borde de ≥10% de las páginas = encabezado/pie

# ================== UTILIDADES ==================
# Marcador "[PÁGINA n]" escrito en los .txt por la celda anterior
PAG_RE   = re.compile(r"^\[PÁGINA (\d+)\]\n", re.M)
# Token numérico (cifras, porcentajes, rangos, montos)
NUM_RE   = re.compile(r"^[\(\$]?\d[\d.,%/\-–)]*$")
# Bloque que es solo un número de página ("12", "iv", "pág. 3", "3 de 120")
SOLO_NUM = re.compile(r"^(?:p[áa]g(?:ina)?\.?\s*)?(?:\d{1,4}|[ivxlcdm]{1,6})(?:\s*(?:de|/)\s*\d+)?$", re.I)
# Puntos guía típicos de un índice (".....12")
LIDER    = re.compile(r"(?:\.\s?){4,}|…{2,}")          # puntos guía de índices ".....12"
# Encabezados de índice y de listas de tablas, figuras, siglas, etc.
HEAD_IDX = re.compile(
    r"^\s*(índice|indice|contenido|contenidos|tabla de contenidos?|"
    r"(lista|índice) de (tablas|gráficos|graficos|figuras|cuadros|mapas|siglas))\b", re.I)

# Lee un .txt y devuelve una lista de tuplas (número de página, texto)
def leer_paginas(path):
    partes = PAG_RE.split(path.read_text(encoding="utf-8"))
    return [(int(partes[i]), partes[i + 1].strip()) for i in range(1, len(partes) - 1, 2)]

# Divide el texto de una página en bloques (cada línea no vacía es un bloque)
def bloques(texto):
    return [b.strip() for b in re.split(r"\n+", texto) if b.strip()]

# Minúsculas y dígitos reemplazados por "#": así se comparan encabezados que solo cambian de número
def normalizar(b):
    return re.sub(r"\d+", "#", b.lower()).strip()

# Separa periodo y presidente a partir del nombre de la carpeta
def parse_gobierno(nombre):
    m = re.match(r"(\d{4}-\d{4})\s*-\s*(.+)", nombre)
    return (m.group(1), m.group(2).strip()) if m else (nombre, "")

# ================== 1) TABLAS ==================
# Devuelve (n.º de palabras, proporción de tokens numéricos, puntos finales por cada 100 palabras)
def metricas(texto):
    palabras = texto.split()
    n = len(palabras)
    if n == 0:
        return 0, 0.0, 0.0
    r_num = sum(bool(NUM_RE.match(p)) for p in palabras) / n
    # La prosa tiene muchos puntos finales; las tablas casi ninguno
    puntos = len(re.findall(r"[.!?](?:\s|$)", texto)) * 100 / n
    return n, r_num, puntos

# Heurística: es tabla si tiene muchos números y pocas oraciones, o un rótulo "Tabla/Cuadro/Gráfico n"
def es_tabla_texto(texto):
    n, r_num, puntos = metricas(texto)
    if n < 15:
        return False
    # Busca un rótulo de tabla o gráfico al inicio de la página
    rotulo = bool(re.search(r"\b(Tabla|Cuadro|Gráfico|Grafico)\s+\d", texto[:300]))
    return (
        r_num >= UMBRAL_NUMERICO
        or (r_num >= UMBRAL_NUM_MIXTO and puntos < MAX_PUNTOS_X100)
        or (rotulo and r_num >= 0.10 and puntos < 3)
    )

# Opcional (USAR_PDF_TABLAS): usa find_tables() de PyMuPDF y marca páginas donde las tablas ocupan ≥20 % del área
def paginas_con_tablas_pdf(pdf_path):
    import pymupdf
    marcadas = set()
    with pymupdf.open(pdf_path) as doc:
        for i, page in enumerate(doc, 1):
            try:
                tablas = [t for t in page.find_tables().tables
                          if t.row_count >= 3 and t.col_count >= 2]
            except Exception:
                continue
            area = sum((t.bbox[2] - t.bbox[0]) * (t.bbox[3] - t.bbox[1]) for t in tablas)
            if area / (page.rect.width * page.rect.height) >= 0.20:
                marcadas.add(i)
    return marcadas

# ================== 2) ÍNDICES, ENCABEZADOS Y PIES ==================
# Una página es índice si tiene ≥3 puntos guía, o si inicia con un encabezado de índice y tiene al menos uno
def es_indice(texto):
    bl = bloques(texto)
    if not bl:
        return False
    lideres = len(LIDER.findall(texto))
    return lideres >= 3 or (bool(HEAD_IDX.match(bl[0])) and lideres >= 1)

def bloques_repetidos(paginas):
    """Bloques cortos que aparecen en los bordes de muchas páginas (dígitos normalizados)."""
    n = len(paginas)
    if n < 10:
        return set()
    cont = Counter()
    for _, t in paginas:
        bl = bloques(t)
        bordes = bl[:2] + bl[-2:]
        for b in {normalizar(x) for x in bordes if len(x.split()) < 15}:
            cont[b] += 1
    return {b for b, c in cont.items() if b and c / n >= UMBRAL_REPETIDO}

# ================== 3-4) PÁRRAFOS Y ORACIONES ==================
# Abreviaturas cuyo punto NO marca el fin de una oración
ABREV = r"(?:Dra?|Sra?|No|Nº|N°|núm|Arts?|arts?|págs?|pp|Fig|Cap|cap|vs|Vol|vol|Ed|ed|Av|Cra|Cll|Inc|Ltda|EE|UU|aprox|ej|Mg|Ing|Lic|Prof|Gral|Cel|Dir|Min)"

# Separa un texto en oraciones, protegiendo abreviaturas e iniciales
def separar_oraciones(texto):
    t = re.sub(rf"\b({ABREV})\.", r"\1§", texto)                  # protege abreviaturas
    t = re.sub(r"\b([A-ZÁÉÍÓÚÑ])\.(?=\s)", r"\1§", t)            # protege iniciales (J. M.)
    partes = re.split(r"(?<=[.!?…])\s+(?=[A-ZÁÉÍÓÚÑ¿¡«“\"(\[\d])", t)
    return [p.replace("§", ".").strip() for p in partes if p.strip()]

# Divide un párrafo largo en trozos de tamaño parejo (< MAX_CARACTERES), cortando en límites de oración
def dividir_por_oraciones(texto):
    if len(texto) <= MAX_CARACTERES:
        return [texto]
    oraciones = separar_oraciones(texto)
    n = math.ceil(len(texto) / MAX_CARACTERES)
    objetivo = len(texto) / n                       # trozos de tamaño parejo
    trozos, actual = [], ""
    for o in oraciones:
        candidato = f"{actual} {o}".strip()
        if actual and (len(candidato) > MAX_CARACTERES or len(actual) >= objetivo):
            trozos.append(actual)
            actual = o
        else:
            actual = candidato
    if actual:
        trozos.append(actual)
    # un trozo final demasiado corto se pega al anterior
    if len(trozos) > 1 and len(trozos[-1].split()) < MIN_PALABRAS:
        ultimo = trozos.pop()
        trozos[-1] += " " + ultimo
    return [p for t in trozos for p in limitar_segmentos(t, MAX_CARACTERES)]

def unir_continuaciones(items):
    """Un bloque que empieza en minúscula continúa al anterior (salto de línea/página)."""
    out = []
    for it in items:
        if out and it["texto"][:1].islower():
            out[-1]["texto"] += " " + it["texto"]
            out[-1]["pagina_fin"] = it.get("pagina_fin", it["pagina"])
        else:
            out.append(dict(it))
    return out

# ================== PROCESAMIENTO ==================
# Mapa (periodo, documento) -> ruta del PDF; sirve para recuperar el nombre del presidente
pdfs = {(p.parent.name[:9], p.stem): p for p in CARPETA_PDFS.rglob("*.pdf")}

# registros: segmentos finales | excluidas: páginas descartadas | resumen: métricas por documento
registros, excluidas, resumen = [], [], []
candidatas_tabla = []

# Procesa cada documento .txt
for txt_path in sorted(CARPETA_TXT.glob("*.txt")):
    periodo, documento = txt_path.stem.split("__", 1)
    pdf = pdfs.get((periodo, documento))
    presidente = parse_gobierno(pdf.parent.name)[1] if pdf else ""

    # Lee las páginas y calcula los bloques repetidos (encabezados/pies) del documento
    paginas = leer_paginas(txt_path)
    tablas_pdf = paginas_con_tablas_pdf(pdf) if (USAR_PDF_TABLAS and pdf) else set()
    repetidos = bloques_repetidos(paginas)

    # Páginas de índice o tabla se excluyen completas; en las demás se filtran bloques de ruido
    items = []
    n_tabla = n_indice = 0
    for num, texto in paginas:
        if not texto:
            continue
        n, r_num, puntos = metricas(texto)
        motivo = None
        if es_indice(texto):
            motivo = "indice"; n_indice += 1
        elif num in tablas_pdf or es_tabla_texto(texto):
            n_tabla += 1
            candidatas_tabla.append({"periodo": periodo, "documento": documento, "pagina": num, "ratio_numerico": r_num, "vista_previa": texto[:300]})
            if EXCLUIR_PAGINAS_TABLA:
                motivo = "tabla"
        if motivo:
            excluidas.append({"periodo": periodo, "documento": documento, "pagina": num,
                              "motivo": motivo, "ratio_numerico": round(r_num, 2),
                              "puntos_x100_palabras": round(puntos, 1),
                              "vista_previa": texto[:120].replace("\n", " ")})
            continue

        for b in bloques(texto):
            if len(b.split()) < 15 and normalizar(b) in repetidos:
                continue                      # encabezado / pie de página
            if SOLO_NUM.match(b) or LIDER.search(b):
                continue                      # número de página suelto o línea de índice
            items.append({"pagina": num, "pagina_fin": num, "texto": b})

    # Une bloques que quedaron cortados por un salto de línea o de página
    items = unir_continuaciones(items)

    # Descartar bloques < 20 palabras
    antes = len(items)
    items = [it for it in items if len(it["texto"].split()) >= MIN_PALABRAS]
    desc_cortos = antes - len(items)

    # Segmentar: los > 2.000 caracteres se dividen por oraciones
    n_div = 0
    for it in items:
        trozos = dividir_por_oraciones(it["texto"])
        n_div += len(trozos) > 1
        for k, t in enumerate(trozos):
            registros.append({
                "periodo": periodo, "presidente": presidente, "documento": documento,
                "pagina": it["pagina"], "pagina_fin": it.get("pagina_fin", it["pagina"]), "parte": k + 1, "de_partes": len(trozos),
                "n_palabras": len(t.split()), "n_caracteres": len(t), "texto": t,
            })

    # Métricas de la segmentación por documento
    resumen.append({"periodo": periodo, "documento": documento, "paginas": len(paginas),
                    "pag_tabla": n_tabla, "pag_indice": n_indice,
                    "bloques_<20_palabras": desc_cortos,
                    "parrafos_divididos": n_div,
                    "segmentos_finales": sum(1 for r in registros
                                             if r["periodo"] == periodo and r["documento"] == documento)})
    print(f"✔ {periodo} | {documento}: {n_tabla} tablas, {n_indice} índice, "
          f"{desc_cortos} bloques cortos")

# ================== GUARDAR ==================
with open(CARPETA_CORPUS / "parrafos.jsonl", "w", encoding="utf-8") as f:
    # Asigna un id único a cada segmento
    for i, r in enumerate(registros, 1):
        r["id"] = i
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

pd.DataFrame(excluidas).to_csv(CARPETA_CORPUS / "paginas_excluidas.csv",
                               index=False, encoding="utf-8-sig")
df_res = pd.DataFrame(resumen)
df_res.to_csv(CARPETA_CORPUS / "resumen_segmentacion.csv", index=False, encoding="utf-8-sig")
print("\n", df_res.to_string(index=False))
print(f"\nTotal de segmentos: {len(registros)}  →  {(CARPETA_CORPUS / 'parrafos.jsonl').resolve()}")
pd.DataFrame(candidatas_tabla).to_csv(CARPETA_CORPUS / "paginas_candidatas_tabla.csv", index=False)
assert all(r["n_caracteres"] <= MAX_CARACTERES for r in registros)


# Parte IV · Etiquetado temático por diccionario de palabras clave

Se asigna un tema a cada segmento contando coincidencias con un diccionario de patrones.

## IV.1 Diccionario de temas y clasificación de segmentos

La celda contiene:

- el **diccionario `TEMAS`** (expresiones regulares sobre texto sin tildes),
- la función **`clasificar`**, que asigna tema principal y secundario según el número de coincidencias (`MIN_HITS`, `RATIO_SECUNDARIO`),
- los **resúmenes** por tema y por gobierno, el análisis de términos frecuentes en "Otros" y una muestra de 100 filas para validación manual.

Salida: `segmentos_clasificados.csv`

In [ ]:
# unicodedata: quitar tildes | CountVectorizer: descubrir términos frecuentes entre los segmentos sin tema
import re, unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer   # pip install scikit-learn openpyxl

# ================== CONFIGURACIÓN ==================
ENTRADA     = Path("corpus/parrafos.jsonl")
SALIDA_XLSX = Path("corpus/segmentos_clasificados.xlsx")
SALIDA_CSV  = Path("corpus/segmentos_clasificados.csv")

# Parámetros de la clasificación por palabras clave
MIN_HITS         = 2     # coincidencias mínimas para asignar un tema (si no: "Otros")
RATIO_SECUNDARIO = 0.6   # el 2.º tema debe tener ≥60% de las coincidencias del 1.º
N_VALIDACION     = 10  # por combinación de periodo y tema; no total   # filas de muestra para revisar a mano

# ================== DICCIONARIO DE TEMAS ==================
# Patrones sobre texto SIN tildes y en minúsculas (ñ -> n). \w* = cualquier terminación.
TEMAS = {
 "Educación": [r"\beducaci\w+", r"\beducativ\w+", r"\bescolar\w*", r"\bescuel\w+", r"\bcolegi\w+",
    r"\bdocentes?\b", r"\bmaestr[oa]s?\b", r"\bestudiantes?\b", r"\buniversidad\w*", r"\buniversitari\w+",
    r"\bsena\b", r"\bicetex\b", r"\bpedagog\w+", r"\bensenanza\b", r"\baprendizaje\w*", r"\bcurricul\w+",
    r"\balfabetiz\w+", r"\banalfabet\w+", r"\bdesercion\b", r"\bprimaria\b", r"\bsecundaria\b",
    r"\bbachiller\w*", r"\bjornada unica\b", r"\bformacion (?:tecnica|profesional|para el trabajo)\b"],
 "Vivienda": [r"\bvivienda\w*", r"\bhabitacional\w*", r"\bhabitat\b", r"\bvis\b", r"\bvip\b",
    r"\bmi casa ya\b", r"\bhipotecari\w+", r"\bupac\b", r"\bleasing habitacional\b",
    r"\bmejoramiento de barrios\b", r"\bdeficit (?:cuantitativo|cualitativo|habitacional)\b",
    r"\basentamientos?\b", r"\bbarrios? (?:subnormales|informales)\b"],
 "Empleo": [r"\bempleos?\b(?!\s+de\s+(?:los|las|la|el)?\s*(?:recursos|fondos|medios|fuerza))",
    r"\bdesempleo\b", r"\bsubempleo\b", r"\bempleabilidad\b", r"\bocupad[oa]s?\b", r"\btasa de ocupacion\b",
    r"\bmercado laboral\b", r"\blaboral\w*", r"\btrabajadores?\b", r"\btrabajo decente\b",
    r"\binformalidad\b", r"\bsalari\w+", r"\bsindical\w*", r"\bpoblacion economicamente activa\b"],
 "Salud": [r"\bsalud\b", r"\bhospital\w*", r"\bsanitari\w+", r"\bmedic\w+", r"\benfermedad\w*",
    r"\bmortalidad\b", r"\bvacun\w+", r"\bepidemi\w+", r"\bpandemi\w+", r"\bnutricion\w*",
    r"\bdesnutricion\b", r"\bregimen (?:subsidiado|contributivo)\b", r"\beps\b", r"\bips\b", r"\bpacientes?\b"],
 "Seguridad y justicia": [r"\bseguridad (?:ciudadana|humana|nacional|publica)\b", r"\bcriminal\w*",
    r"\bdelit\w+", r"\bhomicidio\w*", r"\bpolicia\w*", r"\bfuerza publica\b", r"\bfuerzas militares\b",
    r"\bnarcotrafico\b", r"\bjusticia\b", r"\bjudicial\w*", r"\bcarcel\w*", r"\bpenitenciari\w+",
    r"\bcorrupcion\b", r"\bterroris\w+", r"\bcrimen\w*", r"\bimpunidad\b", r"\bsecuestr\w+", r"\bconvivencia\b"],
 "Paz, conflicto y víctimas": [r"\bpaz\b", r"\bvictimas?\b", r"\bconflicto armado\b", r"\bposconflicto\b",
    r"\bdesplazad\w+", r"\breparacion\w*", r"\brestitucion\b", r"\bdesmovilizacion\b", r"\breintegracion\b",
    r"\breincorporacion\b", r"\bgrupos armados\b", r"\bfarc\b", r"\bparamilitar\w*", r"\bacuerdo de paz\b"],
 "Infraestructura y transporte": [r"\binfraestructura\w*", r"\bvias?\b", r"\bcarretera\w*", r"\btransporte\w*",
    r"\bmovilidad\b", r"\baeropuerto\w*", r"\bpuertos?\b", r"\bferrovia\w*", r"\bferrocarril\w*",
    r"\bconcesion\w*", r"\b4g\b", r"\btroncal\w*", r"\bmetro\b", r"\bvial\w*", r"\blogistic\w+"],
 "Medio ambiente y cambio climático": [r"\bambient\w+", r"\bclimatic\w+", r"\bcambio climatico\b",
    r"\bbiodiversidad\b", r"\bdeforestacion\b", r"\bbosques?\b", r"\becosistem\w+", r"\bemisiones\b",
    r"\bgases de efecto invernadero\b", r"\bcarbono\b", r"\bresiduos\b", r"\bareas protegidas\b",
    r"\brecursos naturales\b", r"\bcuencas?\b", r"\bcontaminacion\b", r"\breciclaj\w+",
    r"\bgestion del riesgo\b", r"\bdesastres?\b"],
 "Agua potable y saneamiento": [r"\bacueducto\w*", r"\balcantarillado\b", r"\bagua potable\b",
    r"\bsaneamiento\b", r"\bresiduos solidos\b", r"\bplantas? de tratamiento\b"],
 "Agro y desarrollo rural": [r"\bagropecuari\w+", r"\bagricol\w+", r"\bagricultur\w+", r"\brural\w*",
    r"\bcampesin\w+", r"\bcultivos?\b", r"\bganader\w+", r"\bcatastro\b", r"\bbaldios\b",
    r"\bseguridad alimentaria\b", r"\bsoberania alimentaria\b", r"\bderecho humano a la alimentacion\b",
    r"\bpesca\w*", r"\bcafe\w*", r"\briego\b", r"\bagroindustri\w+", r"\breforma agraria\b", r"\bagrari\w+"],
 "Pobreza y protección social": [r"\bpobreza\b", r"\bpobres?\b", r"\bvulnerab\w+", r"\bproteccion social\b",
    r"\bsubsidios?\b", r"\btransferencias? monetarias?\b", r"\bfamilias en accion\b", r"\bsisben\b",
    r"\bred unidos\b", r"\bpension\w*", r"\binclusion social\b", r"\bprimera infancia\b", r"\bicbf\b",
    r"\bninez\b", r"\badulto mayor\b", r"\bdiscapacidad\b"],
 "Equidad, género y grupos étnicos": [r"\bgenero\b", r"\bmujer(?:es)?\b", r"\bequidad\b", r"\bdesigualdad\w*",
    r"\betnic\w+", r"\bindigena\w*", r"\bafrocolombian\w+", r"\bdiscriminacion\b", r"\blgbt\w*", r"\bdiversidad\b"],
 "Ciencia, tecnología e innovación": [r"\bciencia\b", r"\btecnologi\w+", r"\binnovaci\w+", r"\binvestigaci\w+",
    r"\bcolciencias\b", r"\bminciencias\b", r"\bdigital\w*", r"\bconectividad\b", r"\binternet\b",
    r"\btics?\b", r"\bbanda ancha\b", r"\btelecomunicaci\w+", r"\bgobierno en linea\b", r"\binteligencia artificial\b"],
 "Energía y minería": [r"\benergi\w+", r"\belectric\w+", r"\bminer\w+", r"\bmineroenergetic\w+",
    r"\bhidrocarburo\w*", r"\bpetroleo\b", r"\bgas natural\b", r"\bcarbon\b", r"\brenovable\w*",
    r"\bsolar\b", r"\beolic\w+", r"\bregalias\b", r"\becopetrol\b", r"\bminas?\b", r"\bsubsuelo\b"],
 "Competitividad, industria, comercio y turismo": [r"\bcompetitiv\w+", r"\bindustri\w+", r"\bexportaci\w+",
    r"\bimportaci\w+", r"\bcomercio\b", r"\bempresa\w*", r"\bempresarial\w*", r"\bemprendimiento\w*",
    r"\bpymes?\b", r"\bmipymes?\b", r"\binversion extranjera\b", r"\btlc\b", r"\baranceles?\b",
    r"\bzonas francas?\b", r"\bproductividad\b", r"\binternacionalizaci\w+", r"\bturismo\b", r"\bturistic\w+"],
 "Macroeconomía y finanzas públicas": [r"\bfiscal\w*", r"\bpresupuest\w+", r"\bdeuda\b", r"\bpib\b",
    r"\binflaci\w+", r"\btributari\w+", r"\bimpuestos?\b", r"\bgasto publico\b", r"\bbalanza de pagos\b",
    r"\btasa de cambio\b", r"\bregla fiscal\b", r"\bahorro\b", r"\binversion publica\b",
    r"\bsistema general de participaciones\b", r"\bsgp\b", r"\bcrecimiento economico\b",
    r"\bmacroeconom\w+", r"\bbanco de la republica\b"],
 "Gobierno, descentralización y participación": [r"\bdescentraliz\w+", r"\bentidades territoriales\b",
    r"\bmunicipios?\b", r"\bordenamiento territorial\b", r"\bparticipacion ciudadana\b", r"\btransparencia\b",
    r"\brendicion de cuentas\b", r"\bmodernizacion del estado\b", r"\badministracion publica\b",
    r"\bgestion publica\b", r"\bbuen gobierno\b", r"\bcontrol social\b", r"\bservicio civil\b"],
 "Cultura, deporte y recreación": [r"\bcultur\w+", r"\bpatrimonio\b", r"\bdeporte\w*", r"\bdeportiv\w+",
    r"\brecreaci\w+", r"\bbibliotec\w+", r"\bmuseos?\b", r"\bartistic\w+", r"\bartes\b",
    r"\bcreativ\w+", r"\beconomia naranja\b", r"\blectura\b"],
}
# Compila un regex por tema uniendo todos sus patrones con OR
TEMAS_RX = {t: re.compile("|".join(f"(?:{p})" for p in ps)) for t, ps in TEMAS.items()}
ORDEN = {t: i for i, t in enumerate(TEMAS)}          # desempate: orden del diccionario
# Etiqueta para segmentos sin coincidencias suficientes
OTROS = "Otros / sin tema claro"

# ================== FUNCIONES ==================
# Pasa a minúsculas y quita tildes (ñ -> n) para que coincida con los patrones del diccionario
def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFKD", t.lower()) if not unicodedata.combining(c))

# Asigna tema principal y (si aplica) secundario según el número de coincidencias de palabras clave
def clasificar(texto):
    t = sin_tildes(texto)
    # Guarda las coincidencias encontradas en cada tema
    hallados = {}
    for tema, rx in TEMAS_RX.items():
        m = [x.group(0) for x in rx.finditer(t)]
        if m:
            hallados[tema] = m
    # Ordena los temas por coincidencias (descendente); si empatan, por orden del diccionario
    ranking = sorted(hallados.items(), key=lambda kv: (-len(kv[1]), ORDEN[kv[0]]))
    # Sin coincidencias suficientes el segmento queda como "Otros"
    if not ranking or len(ranking[0][1]) < MIN_HITS:
        return pd.Series({"tema_principal": OTROS, "tema_secundario": "", "puntaje": 0, "palabras_clave": ""})
    tema1, m1 = ranking[0]
    # El tema secundario solo se asigna si tiene suficientes coincidencias respecto al principal
    tema2 = ""
    if len(ranking) > 1 and len(ranking[1][1]) >= MIN_HITS and len(ranking[1][1]) >= RATIO_SECUNDARIO * len(m1):
        tema2 = ranking[1][0]
    # Las 5 palabras clave más frecuentes del tema principal
    claves = ", ".join(w for w, _ in Counter(m1).most_common(5))
    return pd.Series({"tema_principal": tema1, "tema_secundario": tema2,
                      "puntaje": len(m1), "palabras_clave": claves})

ILEGALES = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")   # openpyxl falla con estos caracteres

# ================== CLASIFICACIÓN ==================
# Carga los segmentos y elimina caracteres de control que rompen la exportación a Excel
df = pd.read_json(ENTRADA, lines=True)
df["texto"] = df["texto"].map(lambda t: ILEGALES.sub("", t))
# Aplica el clasificador a cada segmento y agrega las columnas resultantes
df = pd.concat([df, df["texto"].apply(clasificar)], axis=1)

# Columnas que se guardan en el archivo de salida
cols = ["id", "periodo", "presidente", "documento", "pagina", "pagina_fin", "texto",
        "tema_principal", "tema_secundario", "puntaje", "palabras_clave"]
salida = df[cols]

# ================== RESÚMENES ==================
# Cantidad y porcentaje de segmentos por tema principal
resumen = (df.tema_principal.value_counts().rename_axis("tema").reset_index(name="segmentos"))
resumen["porcentaje"] = (100 * resumen.segmentos / len(df)).round(1)

# % de cada tema dentro de cada gobierno (cada columna suma 100)
por_periodo = (pd.crosstab(df.tema_principal, df.periodo, normalize="columns") * 100).round(1)
por_periodo = por_periodo.reset_index().rename(columns={"tema_principal": "tema"})

# Descubrimiento de temas: términos más frecuentes entre los segmentos sin tema
# Stopwords básicas en español (sin tildes) para el análisis de términos
STOP = set("""de la el en y a los las del se que por un una con para su al lo como mas o pero sus le ya este si
porque esta entre cuando muy sin sobre tambien me hasta hay donde quien desde todo nos durante todos uno les ni
contra otros ese eso ante ellos e esto mi antes algunos unos yo otro otras otra tanto esa estos mucho nada muchos
cual poco ella estar estas algunas algo nosotros ser son es fue han sera sea seran ha mediante cada dos forma
debe deben puede pueden asi parte partir nacional nacionales colombia pais""".split())

# Textos sin tema, normalizados sin tildes
otros_txt = df.loc[df.tema_principal == OTROS, "texto"].map(sin_tildes)
# Tabla vacía por defecto (si hay muy pocos segmentos "Otros")
terminos = pd.DataFrame(columns=["termino", "frecuencia"])
if len(otros_txt) > 20:
    # Unigramas y bigramas frecuentes: sugieren palabras clave o temas que faltan en el diccionario
    cv = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.3, stop_words=list(STOP))
    X = cv.fit_transform(otros_txt)
    terminos = (pd.Series(X.sum(axis=0).A1, index=cv.get_feature_names_out())
                  .sort_values(ascending=False).head(80)
                  .rename_axis("termino").reset_index(name="frecuencia"))

# Muestra para validar a mano
muestra = muestra_validacion(df[cols], por_estrato=N_VALIDACION)
ruta_revision = Path("corpus/validacion_manual.csv")
# Nunca sobrescribir el trabajo de revisión humana.
if not ruta_revision.exists():
    muestra.to_csv(ruta_revision, index=False, encoding="utf-8-sig")
else:
    print("Se conserva validacion_manual.csv; comprobar compatibilidad si cambió el corpus.")

# ================== GUARDAR ==================
salida.to_csv(SALIDA_CSV, index=False, encoding="utf-8-sig")

# Excel con varias hojas: datos, resúmenes, términos y muestra de validación
with pd.ExcelWriter(SALIDA_XLSX, engine="openpyxl") as xw:
    salida.to_excel(xw, sheet_name="segmentos", index=False)
    resumen.to_excel(xw, sheet_name="resumen_temas", index=False)
    por_periodo.to_excel(xw, sheet_name="temas_por_gobierno_%", index=False)
    terminos.to_excel(xw, sheet_name="terminos_en_otros", index=False)
    muestra.to_excel(xw, sheet_name="muestra_validacion", index=False)

    # Formato de las hojas principales: ancho de columnas, fila fija y filtros
    anchos = {"A": 8, "B": 11, "C": 22, "D": 32, "E": 8, "F": 110, "G": 34, "H": 34, "I": 9, "J": 40}
    for hoja in ("segmentos", "muestra_validacion"):
        ws = xw.sheets[hoja]
        for col, w in anchos.items():
            ws.column_dimensions[col].width = w
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

print(resumen.to_string(index=False))
print(f"\nSin tema claro: {(df.tema_principal == OTROS).mean():.1%}")
print("Archivo:", SALIDA_XLSX.resolve())

# Parte V · Preparación de texto (preprocesamiento NLP)

Se carga el corpus clasificado y se prepara el texto para el análisis: limpieza de caracteres, normalización, tokenización y eliminación de stopwords.

In [ ]:
# Cargar el archivo CSV
df = pd.read_csv("corpus/segmentos_clasificados.csv")
# Mostrar las primeras filas
# Dimensiones del DataFrame (segmentos, columnas)
print(df.shape)
df

### V.1 Eliminación de caracteres no alfabéticos

In [ ]:
# Se conserva el original para POS y NER; normalización única para clasificar.
df["texto_letras"] = df["texto"].map(preparar_texto)


### V.2 Normalización a minúsculas

In [ ]:
#Normalización de texto (Convierte todo el texto a minúsculas)
df["texto_norma"] = df["texto_letras"].str.lower()
#texto_norma = df["texto_norma"]
#texto_norma
# Vista previa del resultado
df[["periodo","texto_norma","tema_principal"]]


In [ ]:
# NLTK: tokenizador y lista de stopwords en español
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Descarga de recursos (stopwords y modelo de tokenización "punkt")
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')  # necesario en algunas versiones de NLTK

### V.3 Preservación del contexto
Se conservan stopwords y negaciones en la entrada del clasificador. POS y NER reciben siempre `texto`, con mayúsculas y puntuación.


In [ ]:
# No eliminar stopwords antes de POS/NER; conservar negaciones y contexto.
df["Texto_sin_SW"] = df["texto_norma"]


### V.4 Columna final de texto limpio

In [ ]:
# Transferencia de valores para tener el proceso claro en mi cabeza
df["clean_text"] = df["Texto_sin_SW"]



In [ ]:
#Limpieza del texto finalizado
# Subconjunto con las columnas que se usarán en el análisis y el modelado
Texto_limpio = df[["periodo","clean_text","tema_principal"]]
Texto_limpio

### V.5 Guardar el DataFrame en segmentos clasificados limpios y se Escoge las columnas de interes.

In [ ]:
from pathlib import Path

# Carpeta de salida del corpus procesado
SALIDA = Path("corpus")
SALIDA.mkdir(exist_ok=True)

# CSV (el más liviano y compatible)
df.to_csv(SALIDA / "segm_clasificados_limpios.csv", index=False, encoding="utf-8-sig")


print("Guardado en:", (SALIDA / "segm_clasificados_limpios.csv").resolve())

In [ ]:
# Mantener trazabilidad y texto original en todas las etapas.
columnas = ["id", "periodo", "presidente", "documento", "pagina", "pagina_fin", "texto", "clean_text", "tema_principal", "tema_secundario", "puntaje", "palabras_clave"]
df[columnas].to_csv(SALIDA / "segm_clasificados_limpios.csv", index=False, encoding="utf-8-sig")
chequeo = pd.read_csv(SALIDA / "segm_clasificados_limpios.csv")
assert chequeo.id.is_unique
assert chequeo.texto.notna().all()
print(chequeo.shape)


# Parte VI · Distribución de clases y exploración semántica

Se revisa el balance entre temas, la distribución por gobierno y las características lingüísticas del corpus (categorías gramaticales, entidades nombradas y palabras más frecuentes).

### VI.1 Distribución de segmentos por tema y por gobierno

Se excluyen los segmentos sin tema claro ("Otros") y se grafica la cantidad de segmentos por tema y un mapa de calor con el peso de cada tema en cada periodo.

In [ ]:
# Manejo de datos y gráficos
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Etiqueta de los segmentos sin tema clasificable
OTROS = "Otros / sin tema claro"

# Carga todo el corpus limpio y separa los segmentos con tema claro (los "Otros" se excluyen)
df_all = pd.read_csv("corpus/segm_clasificados_limpios.csv")
df = df_all[df_all["tema_principal"] != OTROS].copy()

# Reporte de cuántos segmentos se excluyen y cuántos se analizan
print(f"Total de segmentos: {len(df_all)}")
print(f"Sin tema claro (excluidos): {len(df_all) - len(df)} ({(len(df_all) - len(df)) / len(df_all):.1%})")
print(f"Segmentos analizados: {len(df)}\n")

# ---------- Distribución de clases recalculada ----------
dist = (df["tema_principal"].value_counts()
          .rename_axis("tema").reset_index(name="segmentos"))
dist["porcentaje"] = (100 * dist["segmentos"] / dist["segmentos"].sum()).round(1)
print(dist.to_string(index=False))
print(f"\nSuma de porcentajes: {dist['porcentaje'].sum():.1f}")

# Orden de los temas de mayor a menor frecuencia (se reutiliza en las gráficas)
orden = dist["tema"].tolist()

# ---------- Gráfica 1: distribución de segmentos por tema ----------
plt.figure(figsize=(9, 6))
ax = sns.countplot(y="tema_principal", data=df, order=orden, color="#4c72b0")
# Etiqueta cada barra con su conteo y porcentaje
for i, (n, p) in enumerate(zip(dist["segmentos"], dist["porcentaje"])):
    ax.text(n, i, f" {n} ({p}%)", va="center", fontsize=9)
ax.set_xlim(0, dist["segmentos"].max() * 1.18)
plt.title("Distribución de segmentos por tema")
plt.xlabel("Cantidad de segmentos"); plt.ylabel("")
plt.tight_layout(); plt.show()

# ---------- Gráfica 2: temas por gobierno (% dentro de cada periodo) ----------
# % de cada tema dentro de cada periodo (cada columna suma 100)
tabla = pd.crosstab(df["tema_principal"], df["periodo"], normalize="columns") * 100
plt.figure(figsize=(10, 7))
sns.heatmap(tabla.loc[orden], annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={"label": "% del periodo"})
plt.title("Temas por gobierno (% de segmentos con tema)")
plt.xlabel("Periodo"); plt.ylabel("")
plt.tight_layout(); plt.show()

# Verificación: cada columna del heatmap suma 100
print(tabla.sum().round(1))

### VI.2 Verificación por periodo

Cantidad de segmentos con tema "Otros" y su proporción por cada periodo.

In [ ]:
# Segmentos con tema por periodo
print(df.groupby("periodo").size())
# Proporción de segmentos "Otros" por periodo (sobre el total de cada periodo)
print((df_all.groupby("periodo")["tema_principal"]
         .apply(lambda s: (s == OTROS).mean())).round(2))   # % de "Otros" por periodo

In [ ]:
# Escribe aquí tu análisis con spaCy, WordCloud o visualizaciones...
# Descarga el modelo de spaCy para español
!python -m spacy download es_core_news_sm
# Stemmer para agrupar palabras de la misma raíz
from nltk.stem import SnowballStemmer
# spaCy: POS tagging, lematización y reconocimiento de entidades (NER)
import spacy

In [ ]:
# Utilidades estándar
import re, math
from collections import Counter, defaultdict
from pathlib import Path

# Datos y gráficos
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# NLP
import spacy
from spacy import displacy
from nltk.stem import SnowballStemmer
# Nubes de palabras
from wordcloud import WordCloud

### VI.3 Configuración y carga de datos

Parámetros del análisis y validación de las columnas de texto y de clase.

In [ ]:
# Archivo de entrada y nombres de columnas
ENTRADA    = Path("corpus/segm_clasificados_limpios.csv")   # Ajustado para usar la carpeta corpus/
COL_TEXTO  = "clean_text"
COL_CLASE  = None            # p. ej. "tema_principal"; si es None se intenta detectar
COL_NER    = "texto"         # columna con el texto original para NER (se ignora si no existe)
OTROS      = "Otros / sin tema claro"

# Carpeta donde se guardan las figuras
FIGURAS = Path("corpus/figuras"); FIGURAS.mkdir(parents=True, exist_ok=True)
# MUESTRA permite probar el flujo con un subconjunto de segmentos
MUESTRA = None               # p. ej. 2000 para probar rápido
# Categorías gramaticales con contenido semántico que se conservan para el análisis de palabras
POS_CONTENIDO = {"NOUN", "PROPN", "ADJ", "VERB"}
# USAR_STEM: agrupa por raíz | TOP_PALABRAS: máximo de palabras por nube
USAR_STEM, TOP_PALABRAS = True, 100

# Stopwords propias del dominio: palabras muy frecuentes en los PND que no distinguen temas
STOP_DOMINIO = {"colombia", "colombiano", "colombiana", "país", "nacional", "plan", "desarrollo",
                "gobierno", "año", "parte", "ser", "estar", "haber", "hacer", "tener", "poder",
                "deber", "dar", "ir", "contar", "permitir", "lograr", "realizar", "así",
                "nuevo", "mayor", "mejor", "público", "social", "general", "través"}

# Traducción de las etiquetas POS al español para las gráficas
POS_ES = {"NOUN": "Sustantivo", "PROPN": "Nombre propio", "VERB": "Verbo", "ADJ": "Adjetivo",
          "ADP": "Preposición", "DET": "Determinante", "AUX": "Auxiliar", "ADV": "Adverbio",
          "CCONJ": "Conj. coordinante", "SCONJ": "Conj. subordinante", "PRON": "Pronombre",
          "NUM": "Numeral", "X": "Otro", "SYM": "Símbolo", "INTJ": "Interjección", "PART": "Partícula"}

# Carga y validación de los datos
df = pd.read_csv(ENTRADA)
print("Columnas del archivo:", list(df.columns))

# --- Detectar la columna de clases ---
if COL_CLASE is None:
    for c in ["tema_principal", "tema", "clase", "label", "categoria", "categoría"]:
        if c in df.columns:
            COL_CLASE = c; break
assert COL_TEXTO in df.columns, f"No existe la columna {COL_TEXTO}"
assert COL_CLASE in df.columns, "Indica COL_CLASE con el nombre de tu columna de clases"

# Elimina textos vacíos y segmentos sin tema claro; opcionalmente toma una muestra
df = df[df[COL_TEXTO].notna() & (df[COL_TEXTO].str.strip() != "")]
df = df[df[COL_CLASE] != OTROS].reset_index(drop=True)
if MUESTRA:
    df = df.sample(MUESTRA, random_state=42).reset_index(drop=True)

assert COL_NER in df.columns, "Falta texto original: ejecutar de nuevo la sección V"
COL_PARA_NER = COL_NER
usar_original, CONSERVA_MAYUS = True, True
print(f"Segmentos: {len(df)} | POS y NER usarán texto original")


### VI.4 Procesamiento con spaCy
POS, lemas y entidades se calculan con el texto original; el filtrado de palabras ocurre después del análisis lingüístico. Revisar manualmente una muestra de entidades antes de interpretarlas.


In [ ]:
# Modelo en español; se desactiva el parser de dependencias (no se usa) para ir más rápido
nlp = spacy.load("es_core_news_sm", disable=["parser"])
stemmer = SnowballStemmer("spanish")
stop_spacy = nlp.Defaults.stop_words

# Contadores por tema: POS, palabras clave (por raíz), lema más frecuente por raíz y entidades
pos_por_tema    = defaultdict(Counter)
claves_por_tema = defaultdict(Counter)
lemas_por_clave = defaultdict(Counter)
ents_por_tema   = defaultdict(Counter)
lemas_seg = []

# --- Pasada 1: POS + lemas (sobre texto original) ---
for tema, doc in zip(df[COL_CLASE], nlp.pipe(df[COL_NER].tolist(), batch_size=64)):
    lemas = []
    for t in doc:
        # Ignora espacios y signos de puntuación
        if t.is_space or t.is_punct:
            continue
        # Cuenta cada token por categoría gramatical
        pos_por_tema[tema][t.pos_] += 1
        # Conserva solo palabras de contenido (sustantivos, adjetivos, verbos) que no sean stopwords
        if t.pos_ in POS_CONTENIDO and t.is_alpha and not t.is_stop:
            l = t.lemma_.lower()
            if len(l) > 2 and l not in stop_spacy and l not in STOP_DOMINIO:
                # "clave" es la raíz de la palabra (agrupa variantes); luego se muestra su lema más frecuente
                clave = stemmer.stem(l) if USAR_STEM else l
                claves_por_tema[tema][clave] += 1
                lemas_por_clave[clave][l] += 1
                lemas.append(l)
    lemas_seg.append(" ".join(lemas))

# --- Pasada 2: NER (sobre el texto con mayúsculas si existe) ---
ents_seg = []
# Con texto original se exige que la entidad empiece en mayúscula (reduce falsos positivos)
exigir_mayus = COL_PARA_NER == COL_NER and usar_original or CONSERVA_MAYUS
for tema, doc in zip(df[COL_CLASE], nlp.pipe(df[COL_PARA_NER].fillna("").tolist(), batch_size=64)):
    ents = []
    for e in doc.ents:
        txt = re.sub(r"\s+", " ", e.text).strip()
        if len(txt) >= 3 and (not exigir_mayus or txt[0].isupper()):
            ents_por_tema[tema][(txt, e.label_)] += 1
            ents.append(f"{txt} ({e.label_})")
    ents_seg.append(" | ".join(ents))

# Guarda lemas y entidades por segmento y exporta el resultado
df["lemas"] = lemas_seg
df["entidades"] = ents_seg
df.to_csv("corpus/segmentos_spacy.csv", index=False, encoding="utf-8-sig")

### VI.5 Ejemplo de análisis morfológico

In [ ]:
# Ejemplo: análisis token a token del primer segmento (POS, lema y morfología)
doc_ej = nlp(df[COL_NER].iloc[0][:400])
pd.DataFrame([(t.text, t.pos_, POS_ES.get(t.pos_, t.pos_), t.lemma_, str(t.morph))
              for t in doc_ej if not t.is_space],
             columns=["token", "POS", "categoría", "lema", "morfología"]).head(15)

### VI.6 Distribución de categorías gramaticales (POS)

Porcentaje global de cada categoría y comparación entre temas.

In [ ]:
# Matriz POS x tema con conteos, ordenada por frecuencia total
pos_df = pd.DataFrame(pos_por_tema).fillna(0).astype(int)
pos_df = pos_df.loc[pos_df.sum(axis=1).sort_values(ascending=False).index]

# a) Global
glob_pct = (100 * pos_df.sum(axis=1) / pos_df.values.sum()).round(1)
plt.figure(figsize=(9, 5))
ax = sns.barplot(x=glob_pct.values, y=[POS_ES.get(p, p) for p in glob_pct.index], color="#4c72b0")
for i, v in enumerate(glob_pct.values):
    ax.text(v, i, f" {v}%", va="center", fontsize=9)
ax.set_xlim(0, glob_pct.max() * 1.15)
plt.title("Distribución de POS tags"); plt.xlabel("% de tokens"); plt.ylabel("")
plt.tight_layout(); plt.savefig(FIGURAS / "pos_global.png", dpi=150); plt.show()

# b) Por clase
# Porcentaje de cada POS dentro de cada tema (cada columna suma 100)
pos_pct = (pos_df / pos_df.sum() * 100).round(1)
pos_pct.index = [POS_ES.get(p, p) for p in pos_pct.index]
plt.figure(figsize=(12, 7))
sns.heatmap(pos_pct, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% de tokens de la clase"})
plt.title("Distribución de POS tags por clase"); plt.xlabel(""); plt.ylabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.savefig(FIGURAS / "pos_por_tema.png", dpi=150); plt.show()

### VI.7 Entidades nombradas (NER)

Entidades más frecuentes de todo el corpus, por tipo (personas, organizaciones, lugares y misceláneo).

In [ ]:
# Tabla en formato largo: tema, entidad, etiqueta y frecuencia
ents_df = pd.DataFrame(
    [(tema, txt, lab, c) for tema, cnt in ents_por_tema.items() for (txt, lab), c in cnt.items()],
    columns=["tema", "entidad", "etiqueta", "frecuencia"])
ents_df.to_csv("corpus/entidades_por_tema.csv", index=False, encoding="utf-8-sig")

# Frecuencia total de cada entidad en todo el corpus
top = ents_df.groupby(["etiqueta", "entidad"], as_index=False).frecuencia.sum()
print(top.groupby("etiqueta").frecuencia.sum().rename("menciones"))

# Tipos de entidad que detecta spaCy en español
ETIQ = {"PER": "Personas", "ORG": "Organizaciones", "LOC": "Lugares", "MISC": "Misceláneo"}
# Un panel por tipo de entidad con las 15 más frecuentes
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (lab, nombre) in zip(axes.ravel(), ETIQ.items()):
    d = top[top.etiqueta == lab].nlargest(15, "frecuencia").iloc[::-1]
    ax.barh(d["entidad"], d["frecuencia"], color="#4c72b0")
    ax.set_title(nombre); ax.tick_params(axis="y", labelsize=9)
plt.suptitle("Entidades nombradas más frecuentes", fontsize=14)
plt.tight_layout(); plt.savefig(FIGURAS / "ner_top.png", dpi=150); plt.show()

# Segmento con entidades resaltadas
idx = df.index[df["entidades"] != ""]
if len(idx):
    displacy.render(nlp(df.loc[idx[0], COL_PARA_NER]), style="ent", jupyter=True)

### VI.8 Entidades por tema (tablas)

Top 20 de entidades en Educación, Vivienda y Empleo.

In [ ]:
# Temas que se van a comparar
CLASES = ["Educación", "Vivienda", "Empleo"]

# Tabla con las 20 entidades más frecuentes de cada tema
for clase in CLASES:
    print(f"\n=== {clase} ===")
    display(
        ents_df[ents_df.tema == clase]
        .nlargest(20, "frecuencia")
        .reset_index(drop=True)[["entidad", "etiqueta", "frecuencia"]]
    )

### VI.9 Entidades por tema (gráficas Top 10)

In [ ]:
# Un panel por tema con sus 10 entidades más frecuentes
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, clase in zip(axes, CLASES):
    d = (ents_df[ents_df.tema == clase]
         .nlargest(10, "frecuencia")
         .iloc[::-1])
    ax.barh(d["entidad"] + " (" + d["etiqueta"] + ")", d["frecuencia"], color="#4c72b0")
    ax.set_title(clase)
    ax.tick_params(axis="y", labelsize=9)
plt.suptitle("Entidades más frecuentes por clase", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURAS / "ner_educacion_vivienda_empleo.png", dpi=150)
plt.show()

### VI.10 Nubes de palabras por tema

Palabras clave más frecuentes por tema, agrupadas por raíz y mostradas con su lema más común.

In [ ]:
CLASES = ["Educación", "Vivienda", "Empleo"]

# Devuelve {lema: frecuencia} de las palabras clave más frecuentes de un tema
def frecuencias(tema, n=TOP_PALABRAS):
    out = Counter()
    for clave, c in claves_por_tema[tema].most_common(n):
        out[lemas_por_clave[clave].most_common(1)[0][0]] += c
    return dict(out)

# Una nube de palabras por tema
fig, axes = plt.subplots(1, len(CLASES), figsize=(6 * len(CLASES), 4.2))

for ax, tema in zip(axes, CLASES):
    freqs = frecuencias(tema)
    if freqs:
        wc = WordCloud(width=700, height=400, background_color="white", colormap="viridis",
                       max_words=TOP_PALABRAS, random_state=42).generate_from_frequencies(freqs)
        ax.imshow(wc, interpolation="bilinear")
        # Guarda cada nube como imagen independiente
        wc.to_file(FIGURAS / f"nube_{re.sub(r'[^A-Za-z0-9]+', '_', tema)}.png")
    else:
        ax.text(0.5, 0.5, "Sin datos", ha="center", va="center")
    ax.set_title(f"{tema}  (n={(df[COL_CLASE] == tema).sum()})", fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURAS / "nubes_educacion_vivienda_empleo.png", dpi=130)
plt.show()

### VI.11 Cobertura y comparación entre periodos
Los porcentajes sobre segmentos clasificados son condicionales a la cobertura del diccionario. Se presentan también Otros y una medida por palabras. Asignar todas las palabras al tema principal sigue siendo una aproximación; no mide gasto ni resultados de política pública.


In [ ]:
base = pd.read_csv("corpus/segm_clasificados_limpios.csv")
base["n_palabras"] = base.texto.str.split().str.len()
cobertura = base.assign(clasificado=base.tema_principal.ne(OTROS)).groupby("periodo").agg(segmentos=("id", "size"), cobertura=("clasificado", "mean"))
display(cobertura)
display(pd.crosstab(base.tema_principal, base.periodo, normalize="columns").mul(100).round(2))
peso_palabras = base.pivot_table(index="tema_principal", columns="periodo", values="n_palabras", aggfunc="sum", fill_value=0)
display(peso_palabras.div(peso_palabras.sum(axis=0), axis=1).mul(100).round(2))
cobertura.to_csv("corpus/cobertura_por_periodo.csv")


# Parte VII · Modelado: clasificación de temas

Se entrena un clasificador de texto con TF-IDF y regresión logística. Primero con todos los temas  y luego con un subconjunto de tres clases.

### VII.A.1 Librerías y parámetros

In [ ]:
# Manejo de datos y gráficos
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Aplica TfidfVectorizer, entrena el modelo_18, y evalúa...
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Parámetros: archivo, columna de texto, variable objetivo y mínimo de ejemplos por clase
ENTRADA   = "corpus/segm_clasificados_limpios.csv"
COL_TEXTO = "clean_text"
COL_Y     = "tema_principal"        # ¡¡¡ En caso de implementar columna de sentimiento !!!
OTROS     = "Otros / sin tema claro"
MIN_CLASE = 30                       # clases con menos ejemplos se descartan

### VII.A.2 Carga de datos y partición entrenamiento/prueba

In [ ]:
# Carga y filtrado: sin textos vacíos, sin "Otros" y sin clases con pocos ejemplos
df = reservar_revision(pd.read_csv(ENTRADA))
df = df.drop_duplicates(subset="clean_text").copy()
df = df[df[COL_TEXTO].notna() & (df[COL_TEXTO].str.strip() != "")]
df = df[df[COL_Y] != OTROS]

conteo = df[COL_Y].value_counts()
df = df[df[COL_Y].isin(conteo[conteo >= MIN_CLASE].index)].reset_index(drop=True)

# Distribución de las clases que se conservan
print(df[COL_Y].value_counts())

# Partición 80/20 estratificada (mantiene la proporción de clases) y reproducible
X_train_18, X_test_18, y_train_18, y_test_18 = train_test_split(
    df[COL_TEXTO], df[COL_Y],
    test_size=0.2, stratify=df[COL_Y], random_state=42)

### VII.A.3 Vectorización TF-IDF

In [ ]:
# TF-IDF con unigramas y bigramas; min_df/max_df descartan términos muy raros o muy comunes; sublinear_tf suaviza frecuencias altas
tfidf_18 = TfidfVectorizer(preprocessor=preparar_texto, lowercase=False, ngram_range=(1, 2), min_df=5, max_df=0.5,
                        sublinear_tf=True, max_features=50_000)

X_train_tfidf_18 = tfidf_18.fit_transform(X_train_18)   # aprende vocabulario SOLO con entrenamiento
X_test_tfidf_18  = tfidf_18.transform(X_test_18)        # aplica ese mismo vocabulario a prueba

print("Matriz de entrenamiento:", X_train_tfidf_18.shape)
print("Matriz de prueba:", X_test_tfidf_18.shape)


### VII.A.4 Entrenamiento del modelo

In [ ]:
# Regresión logística; class_weight="balanced" compensa el desbalance entre clases
modelo_18 = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")
modelo_18.fit(X_train_tfidf_18, y_train_18)

### VII.A.5 Evaluación

In [ ]:
# Predice sobre el conjunto de prueba y muestra precisión, recall y F1 por clase
y_pred_18 = modelo_18.predict(X_test_tfidf_18)
print(classification_report(y_test_18, y_pred_18, digits=3, zero_division=0))

### VII.A.6 Matriz de confusión

In [ ]:
# Matriz de confusión en conteos y normalizada por clase real
clases = sorted(y_test_18.unique())
cm = confusion_matrix(y_test_18, y_pred_18, labels=clases)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=clases, yticklabels=clases, ax=axes[0])
axes[0].set_title("Matriz de confusión (conteos)")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=clases, yticklabels=clases, ax=axes[1])
axes[1].set_title("Matriz de confusión (% por clase real)")

# Etiquetas de los ejes y rotación de los nombres de clase
for ax in axes:
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout(); plt.show()

### VII.A.7 Términos más influyentes por clase

In [ ]:
import numpy as np

# Términos con mayor coeficiente por clase: los que más pesan en la predicción
vocab = np.array(tfidf_18.get_feature_names_out())
coefs = modelo_18.coef_
if len(modelo_18.classes_) == 2:                  # caso binario: un solo vector de coeficientes
    coefs = np.vstack([-coefs[0], coefs[0]])

for clase, c in zip(modelo_18.classes_, coefs):
    print(f"{clase:45s} → {', '.join(vocab[np.argsort(c)[-12:][::-1]])}")

### VII.A.8 Predicción con texto nuevo

**Texto ejemplo**

Se clasifican textos nuevos aplicando el mismo preprocesamiento del entrenamiento y respetando el largo de los segmentos del corpus (20 palabras o más y 2.000 caracteres o menos).


| Tema | Texto |
|---|---|
| Educación | "gobierno fortalecerá educación básica media docentes mejores escuelas ampliará cobertura escolar reducirá deserción estudiantes mejorará calidad aprendizaje mediante jornada única formación maestros instituciones educativas país" |
| Vivienda | "promoverá acceso vivienda interés social subsidios habitacionales mejoramiento barrios. programa reducirá déficit cuantitativo cualitativo vivienda, facilitará crédito hipotecario familias menores ingresos legalizará asentamientos informales ciudades" |
| Empleo | "política laboral buscará reducir desempleo informalidad mercado laboral promoverá generación empleo formal empleabilidad jóvenes trabajo decente mejores salarios mayor protección trabajadores aumentar tasa  ocupación" |
| Mixto | "educación técnica mejora empleabilidad jóvenes también requiere vivienda cerca centros trabajo familias necesitan acceso servicios básicos poder estudiar trabajar estabilidad" |


In [ ]:
# Texto natural: aplicar exactamente la preparación usada durante el entrenamiento.
nuevo = ["La educación técnica mejora la empleabilidad de los jóvenes. También se requiere vivienda cerca de los centros de trabajo y acceso a servicios básicos para sus familias."]
print("Modelo de 18 temas:", modelo_18.predict(tfidf_18.transform(nuevo)))


In [ ]:
probs = modelo_18.predict_proba(tfidf_18.transform(nuevo))[0]
print(pd.Series(probs, index=modelo_18.classes_).sort_values(ascending=False).head(5))
# Son probabilidades del modelo, no una medida calibrada de certeza.


## Vectorización y entrenamiento con 3 clases

### VII.B.1 Librerías y parámetros

In [ ]:
# Manejo de datos, gráficos y modelado
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Parámetros: se trabaja solo con tres temas
ENTRADA   = "corpus/segm_clasificados_limpios.csv"
COL_TEXTO = "clean_text"
COL_Y     = "tema_principal"
CLASES    = ["Educación", "Vivienda", "Empleo"]

### VII.B.2 Carga de datos y partición entrenamiento/prueba

In [ ]:
# Carga y filtra las 3 clases de interés
df = reservar_revision(pd.read_csv(ENTRADA))
df = df.drop_duplicates(subset="clean_text").copy()
df = df[df[COL_TEXTO].notna() & (df[COL_TEXTO].str.strip() != "")]
df = df[df[COL_Y].isin(CLASES)].reset_index(drop=True)

print(df[COL_Y].value_counts())
print(df[COL_Y].value_counts(normalize=True).round(3))

# Partición 80/20 estratificada
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    df[COL_TEXTO], df[COL_Y],
    test_size=0.2, stratify=df[COL_Y], random_state=42)

### VII.B.3 Vectorización TF-IDF y entrenamiento

In [ ]:
# TF-IDF con umbral más bajo (min_df=3) y vocabulario máximo de 30.000 términos
tfidf_3 = TfidfVectorizer(preprocessor=preparar_texto, lowercase=False, ngram_range=(1, 2), min_df=3, max_df=0.5,
                        sublinear_tf=True, max_features=30_000)

X_train_tfidf_3 = tfidf_3.fit_transform(X_train_3)   # vocabulario solo con entrenamiento
X_test_tfidf_3  = tfidf_3.transform(X_test_3)
print("Matriz de entrenamiento:", X_train_tfidf_3.shape)

# Regresión logística con pesos balanceados
modelo_3 = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")
modelo_3.fit(X_train_tfidf_3, y_train_3)

### VII.B.4 Evaluación y matriz de confusión

In [ ]:
# Evaluación sobre el conjunto de prueba
y_pred_3 = modelo_3.predict(X_test_tfidf_3)
print(classification_report(y_test_3, y_pred_3, digits=3, zero_division=0))

# Matriz de confusión: conteos y proporción por clase real
cm = confusion_matrix(y_test_3, y_pred_3, labels=CLASES)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=CLASES, yticklabels=CLASES, ax=axes[0])
axes[0].set_title("Matriz de confusión (conteos)")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASES, yticklabels=CLASES, ax=axes[1])
axes[1].set_title("Matriz de confusión (% por clase real)")
for ax in axes:
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
plt.tight_layout(); plt.show()

### VII.B.5 Términos más influyentes por clase

In [ ]:
# Términos más influyentes por clase
vocab = np.array(tfidf_3.get_feature_names_out())
for clase, c in zip(modelo_3.classes_, modelo_3.coef_):
    print(f"\n{clase}")
    print("  " + ", ".join(vocab[np.argsort(c)[-15:][::-1]]))

### VII.B.6 Análisis de errores

In [ ]:
# Análisis de errores: texto, clase real y clase predicha
res = pd.DataFrame({"texto": X_test_3.values, "real": y_test_3.values, "predicho": y_pred_3})
errores = res[res.real != res.predicho]
# Cantidad de errores y confusiones más frecuentes
print(f"Errores: {len(errores)} de {len(res)} ({len(errores) / len(res):.1%})")
print(errores.groupby(["real", "predicho"]).size().sort_values(ascending=False))

# Muestra textos completos y 8 errores al azar
pd.set_option("display.max_colwidth", 300)
errores.sample(min(8, len(errores)), random_state=1)

# Parte VIII · Evaluaciones complementarias

## VIII.1 Generalización entre periodos
Se deja un gobierno fuera en cada iteración y se eliminan del entrenamiento duplicados exactos normalizados del periodo de prueba. Esto reduce la contaminación entre documentos relacionados, aunque no detecta paráfrasis. Las etiquetas de prueba siguen siendo automáticas: esta evaluación mide transferencia de las reglas entre periodos. El subconjunto manual queda reservado.


In [ ]:
base_modelo = reservar_revision(pd.read_csv("corpus/segm_clasificados_limpios.csv"))
base_modelo = base_modelo[base_modelo.tema_principal.ne(OTROS)].copy()
resultados_periodo = []
for periodo in sorted(base_modelo.periodo.unique()):
    entrenamiento, prueba = separar_periodo(base_modelo, periodo)
    entrenamiento = entrenamiento.drop_duplicates("clean_text")
    if entrenamiento.tema_principal.nunique() < 2 or prueba.empty:
        continue
    pipeline = nuevo_modelo()
    pipeline.fit(entrenamiento.texto, entrenamiento.tema_principal)
    pred = pipeline.predict(prueba.texto)
    reporte = classification_report(prueba.tema_principal, pred, output_dict=True, zero_division=0)
    pd.DataFrame(reporte).T.to_csv(f"corpus/evaluacion_periodo_{periodo}.csv")
    resultados_periodo.append({"periodo": periodo, "n_train": len(entrenamiento), "n_test": len(prueba), "accuracy": reporte["accuracy"], "macro_f1": reporte["macro avg"]["f1-score"], "clases_no_vistas": "; ".join(sorted(set(prueba.tema_principal) - set(entrenamiento.tema_principal)))})
display(pd.DataFrame(resultados_periodo))


## VIII.2 Validación humana independiente
1. Abrir `corpus/validacion_manual.csv`. Cada persona etiqueta el texto original en su columna sin consultar la etiqueta automática ni a la otra persona; para una revisión ciega, ocultar las columnas automáticas.
2. Usar la taxonomía de 18 temas y Otros. Acordar el tema principal por el propósito del fragmento, no por el mero conteo de palabras. Registrar ambigüedad y temas secundarios en observaciones. Resolver discrepancias en `tema_consenso`.
3. No modificar el diccionario ni ajustar hiperparámetros con esta muestra de prueba. Si se usa para mejoras, reservar otra muestra independiente para evaluación final.
4. Los estratos tienen tamaños distintos a la población: las métricas sin ponderar describen esta muestra, no una estimación representativa del corpus. Informar métricas por clase y por periodo.
5. Si cambia la segmentación, los IDs pueden cambiar: archivar la revisión anterior y generar una nueva muestra; no transferir etiquetas por número de ID.

No se generan etiquetas humanas ni resultados ficticios. La evaluación solo se activa al completar el consenso.


In [ ]:
base_humana = pd.read_csv("corpus/segm_clasificados_limpios.csv")
gold = leer_consenso("corpus/validacion_manual.csv", base_humana)
if gold.empty:
    print("Pendiente: completar tema_consenso en corpus/validacion_manual.csv y ejecutar esta celda.")
else:
    revision = pd.read_csv("corpus/validacion_manual.csv")
    print("Acuerdo entre revisores:", acuerdo_revisores(revision))
    print("Diccionario vs. consenso humano (incluye Otros):")
    print(classification_report(gold.tema_consenso, gold.tema_principal, zero_division=0))
    train_humano = reservar_revision(base_humana)
    train_humano = train_humano[train_humano.tema_principal.ne(OTROS)].drop_duplicates("clean_text")
    modelo_validacion = nuevo_modelo()
    modelo_validacion.fit(train_humano.texto, train_humano.tema_principal)
    gold["pred_modelo"] = modelo_validacion.predict(gold.texto)
    print("Modelo vs. consenso humano:")
    print(classification_report(gold.tema_consenso, gold.pred_modelo, zero_division=0))
    # El modelo cerrado no predice Otros: esos fallos deben permanecer visibles.
    for periodo, parte in gold.groupby("periodo"):
        print(periodo)
        print(classification_report(parte.tema_consenso, parte.pred_modelo, zero_division=0))
    gold.to_csv("corpus/evaluacion_humana_detalle.csv", index=False)


# Parte IX · Interpretación y límites

Completar después de ejecutar la versión corregida:

- **Cobertura:** reportar segmentos y palabras conservados, páginas excluidas y Otros por periodo. Auditar páginas candidatas a tabla e índices (que antes registraban cero exclusiones por índice).
- **Resultados temáticos:** describir tres hallazgos sobre educación, vivienda y empleo, con porcentajes que indiquen su denominador y citas identificadas por documento y páginas.
- **Validación:** separar métricas contra diccionario, transferencia entre gobiernos y comparación con consenso humano. No comparar directamente la exactitud de 18 y tres clases: son tareas distintas.
- **Ambigüedad:** el modelo usa tema principal aunque el corpus registra un secundario. Explorar multietiqueta solo tras validar anotaciones suficientes; los empates del diccionario favorecen su orden de definición.
- **Límites:** documentos de diferente extensión y estructura; heurísticas de segmentación; posibles duplicados aproximados; taxonomía y umbrales del diccionario; errores de NER y ausencia de validación humana hasta completarla.
- **Alcance:** la frecuencia de términos expresa presencia discursiva; no permite concluir inversión, cumplimiento ni impacto de las políticas.
